# Chapter 13: Tadpoles and Partial Pooling

## The Core Idea

When data is grouped (students in schools, patients in hospitals, tadpoles in tanks), we have three options:

| Approach | What it does | Problem |
|----------|-------------|--------|
| **Complete pooling** | One parameter for all groups | Ignores group differences |
| **No pooling** | Fully separate parameter per group | Ignores that groups are similar; overfits small groups |
| **Partial pooling** | Parameters drawn from a common distribution | Best of both — shares information across groups |

Partial pooling is what makes a model **multilevel** (or hierarchical). Groups share a **prior whose parameters are estimated from the data** — the prior learns from the data itself.

**Example**: Reed frog tadpoles in 48 experimental tanks. Each tank has a different number of tadpoles and different conditions. We want to estimate the survival rate per tank.

---

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pymc as pm
import arviz as az
from scipy import special

plt.style.use('default')
%matplotlib inline
rng = np.random.default_rng(42)

print(f'PyMC {pm.__version__} | ArviZ {az.__version__}')

PyMC 5.26.1 | ArviZ 0.22.0


## The Data

In [2]:
url = 'https://raw.githubusercontent.com/rmcelreath/rethinking/master/data/reedfrogs.csv'
frogs = pd.read_csv(url, sep=';')
print(frogs.head(10).to_string())
print(f'\nShape: {frogs.shape}')
print(f'Columns: {frogs.columns.tolist()}')

  density;"pred";"size";"surv";"propsurv"
0                     10;"no";"big";9;0.9
1                      10;"no";"big";10;1
2                     10;"no";"big";7;0.7
3                      10;"no";"big";10;1
4                   10;"no";"small";9;0.9
5                   10;"no";"small";9;0.9
6                    10;"no";"small";10;1
7                   10;"no";"small";9;0.9
8                   10;"pred";"big";4;0.4
9                   10;"pred";"big";9;0.9

Shape: (48, 1)
Columns: ['density;"pred";"size";"surv";"propsurv"']


In [3]:
# Each row is one tank
surv     = frogs['surv'].values       # number surviving
density  = frogs['density'].values    # initial number
tank_id  = np.arange(len(frogs))      # tank index 0..47
n_tanks  = len(frogs)
surv_rate = surv / density

print(f'Tanks: {n_tanks}')
print(f'Density range: {density.min()} – {density.max()}')
print(f'Survival rate range: {surv_rate.min():.2f} – {surv_rate.max():.2f}')
print(f'Mean survival rate: {surv_rate.mean():.2f}')

# Visualise raw survival rates
fig, ax = plt.subplots(figsize=(13, 4))
colors = np.where(density <= 10, '#2196F3',
         np.where(density <= 25, '#4CAF50', '#FF9800'))
ax.bar(tank_id, surv_rate, color=colors, alpha=0.8, edgecolor='white')
ax.axhline(surv_rate.mean(), color='black', ls='--', lw=1.5,
           label=f'Grand mean = {surv_rate.mean():.2f}')
ax.set_xlabel('Tank', fontsize=11)
ax.set_ylabel('Survival rate', fontsize=11)
ax.set_ylim(0, 1)
ax.set_title('Raw survival rates across 48 tanks\n'
             'Blue = small (≤10), green = medium (≤25), orange = large (>25)',
             fontsize=11, fontweight='bold')
ax.legend()
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

KeyError: 'surv'

## Three Models

### Model 1: No Pooling
Each tank gets its own independent α. No information shared across tanks.
$$S_i \sim \text{Binomial}(N_i,\ p_i)$$
$$\text{logit}(p_i) = \alpha_{\text{tank}[i]}$$
$$\alpha_j \sim \text{Normal}(0, 1.5) \quad \text{(same fixed prior for all tanks)}$$

### Model 2: Partial Pooling (Multilevel)
Tank parameters drawn from a **common distribution whose parameters are estimated**.
$$S_i \sim \text{Binomial}(N_i,\ p_i)$$
$$\text{logit}(p_i) = \alpha_{\text{tank}[i]}$$
$$\alpha_j \sim \text{Normal}(\bar{\alpha},\ \sigma) \quad \leftarrow \text{adaptive prior}$$
$$\bar{\alpha} \sim \text{Normal}(0, 1.5) \quad \leftarrow \text{hyperprior on the mean}$$
$$\sigma \sim \text{Exponential}(1) \quad \leftarrow \text{hyperprior on the spread}$$

The key difference: `ᾱ` and `σ` are **hyperparameters** — parameters of the distribution of parameters. They are estimated from the data, making the prior adaptive.

In [ ]:
# Model 1: No pooling
with pm.Model() as m_no_pool:
    alpha = pm.Normal('alpha', mu=0, sigma=1.5, shape=n_tanks)
    p     = pm.Deterministic('p', pm.math.invlogit(alpha))
    obs   = pm.Binomial('obs', n=density, p=p[tank_id], observed=surv)

    idata_no_pool = pm.sample(1000, tune=1000, chains=4,
                              target_accept=0.9, random_seed=42, progressbar=True)

print('No pooling model fitted.')

In [ ]:
# Model 2: Partial pooling (multilevel)
with pm.Model() as m_partial:
    # Hyperpriors — learned from data
    alpha_bar = pm.Normal('alpha_bar', mu=0, sigma=1.5)    # mean of tank distribution
    sigma     = pm.Exponential('sigma', lam=1)             # spread of tank distribution

    # Tank-level parameters — drawn from the adaptive prior
    alpha = pm.Normal('alpha', mu=alpha_bar, sigma=sigma, shape=n_tanks)
    p     = pm.Deterministic('p', pm.math.invlogit(alpha))

    obs = pm.Binomial('obs', n=density, p=p[tank_id], observed=surv)

    idata_partial = pm.sample(1000, tune=1000, chains=4,
                              target_accept=0.9, random_seed=42, progressbar=True)

print('Partial pooling model fitted.')
print()
print('Hyperparameters (the learned prior):')
print(az.summary(idata_partial, var_names=['alpha_bar', 'sigma'], hdi_prob=0.89).to_string())

## Shrinkage: The Key Feature of Partial Pooling

Partial pooling **shrinks** estimates toward the grand mean (ᾱ). The amount of shrinkage depends on:
1. **Tank sample size**: small tanks shrink more (less data → rely more on the prior)
2. **σ**: if σ is small (tanks are similar), more shrinkage; if σ is large (tanks are different), less shrinkage

This is sometimes called **regularisation** — the adaptive prior acts like a regulariser, preventing extreme estimates from small-sample tanks.

In [ ]:
# Extract posterior means for tank survival probabilities
p_no_pool = idata_no_pool.posterior['p'].values.reshape(-1, n_tanks).mean(axis=0)
p_partial  = idata_partial.posterior['p'].values.reshape(-1, n_tanks).mean(axis=0)

# Grand mean from partial pooling (on probability scale)
alpha_bar_s = idata_partial.posterior['alpha_bar'].values.flatten()
grand_mean  = special.expit(alpha_bar_s.mean())

fig, ax = plt.subplots(figsize=(13, 5))

# Raw observed rates
ax.scatter(tank_id, surv_rate, color='black', s=30, zorder=5,
           label='Observed rate', alpha=0.7)

# No pooling estimates
ax.scatter(tank_id, p_no_pool, color='steelblue', s=30, marker='s', zorder=4,
           label='No pooling', alpha=0.7)

# Partial pooling estimates
ax.scatter(tank_id, p_partial, color='tomato', s=30, marker='^', zorder=4,
           label='Partial pooling', alpha=0.9)

# Connect no-pool → partial-pool to show shrinkage
for i in range(n_tanks):
    ax.plot([i, i], [p_no_pool[i], p_partial[i]],
            color='gray', lw=0.8, alpha=0.5, zorder=3)

# Grand mean
ax.axhline(grand_mean, color='tomato', ls='--', lw=1.5,
           label=f'Partial pooling grand mean = {grand_mean:.2f}')

# Tank size boundaries
ax.axvline(15.5, color='gray', ls=':', alpha=0.5)
ax.axvline(31.5, color='gray', ls=':', alpha=0.5)
ax.text(7, 0.05, 'Small\n(n≤10)', ha='center', fontsize=9, color='gray')
ax.text(23, 0.05, 'Medium\n(n≤25)', ha='center', fontsize=9, color='gray')
ax.text(39, 0.05, 'Large\n(n>25)', ha='center', fontsize=9, color='gray')

ax.set_xlabel('Tank', fontsize=11)
ax.set_ylabel('Survival probability', fontsize=11)
ax.set_ylim(0, 1)
ax.set_title('Shrinkage: partial pooling pulls estimates toward the grand mean\n'
             'Small tanks shrink more; large tanks shrink less',
             fontsize=11, fontweight='bold')
ax.legend(fontsize=9, loc='lower right')
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

# Quantify shrinkage by tank size
shrinkage = np.abs(p_no_pool - p_partial)
small  = density <= 10
medium = (density > 10) & (density <= 25)
large  = density > 25
print(f'Mean shrinkage by tank size:')
print(f'  Small  (n≤10):  {shrinkage[small].mean():.3f}')
print(f'  Medium (n≤25):  {shrinkage[medium].mean():.3f}')
print(f'  Large  (n>25):  {shrinkage[large].mean():.3f}')

## Why Partial Pooling Wins

### Intuition
- **No pooling** treats each tank as an island. A tank with 1 survivor out of 2 gets an estimate of 0.5 — but with only 2 observations, we shouldn't be confident in that.
- **Partial pooling** says: "the prior for this tank is informed by all other tanks." If most tanks survive at ~75%, the estimate for the 1/2 tank gets pulled toward 75%.

### The adaptive prior does the right thing automatically
- If σ is large (tanks are very different from each other): little shrinkage — the data says groups are genuinely different
- If σ is small (tanks are similar): strong shrinkage — the data says groups share a common rate

### Shrinkage ≠ bias
Partial pooling does introduce some bias for each individual tank, but **reduces variance** so much that overall prediction error (WAIC, LOO) is lower. This is the **bias-variance tradeoff** in action.

In [ ]:
# Simulate to show partial pooling beats no-pooling out-of-sample
# Use the estimated hyperparameters to generate new tanks and see which model predicts better

# Posterior of hyperparameters
alpha_bar_s = idata_partial.posterior['alpha_bar'].values.flatten()
sigma_s     = idata_partial.posterior['sigma'].values.flatten()

print('Estimated hyperparameters (the learned prior over tanks):')
print(f'  ᾱ  = {alpha_bar_s.mean():.3f}  89% CI [{np.percentile(alpha_bar_s,5.5):.3f}, '
      f'{np.percentile(alpha_bar_s,94.5):.3f}]')
print(f'  σ  = {sigma_s.mean():.3f}  89% CI [{np.percentile(sigma_s,5.5):.3f}, '
      f'{np.percentile(sigma_s,94.5):.3f}]')
print()
print(f'Grand mean survival probability: {special.expit(alpha_bar_s.mean()):.2f}')
print()
print('Interpretation:')
print(f'  Tanks vary in log-odds survival with SD ≈ {sigma_s.mean():.2f}')
print(f'  On probability scale this means most tanks survive between '
      f'{special.expit(alpha_bar_s.mean() - sigma_s.mean()):.0%} and '
      f'{special.expit(alpha_bar_s.mean() + sigma_s.mean()):.0%}')

In [ ]:
# Visualise the learned prior over tank survival probabilities
fig, ax = plt.subplots(figsize=(8, 4))

# Sample tank survival rates from the learned prior
n_sim = 5000
idx   = rng.integers(0, len(alpha_bar_s), n_sim)
alpha_new = rng.normal(alpha_bar_s[idx], sigma_s[idx])
p_new     = special.expit(alpha_new)

ax.hist(p_new, bins=50, density=True, color='tomato', alpha=0.7, edgecolor='white')
ax.hist(surv_rate, bins=15, density=True, color='steelblue', alpha=0.5,
        edgecolor='white', label='Observed tank rates')
ax.set_xlabel('Survival probability', fontsize=11)
ax.set_ylabel('Density', fontsize=11)
ax.set_title('Learned prior over tank survival rates\n'
             'Red = samples from learned Normal(ᾱ, σ),  Blue = observed rates',
             fontsize=10, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Key Insights

### 1. Partial pooling is the default for grouped data
Whenever you have data in groups (tanks, schools, patients, countries), partial pooling almost always outperforms both complete pooling and no pooling. Use it as your default.

### 2. The hyperprior learns from the data
The adaptive prior `Normal(ᾱ, σ)` is not fixed — `ᾱ` and `σ` are estimated. This means the model learns *how variable* the groups are, and adjusts shrinkage accordingly.

### 3. Shrinkage is adaptive and automatic
- Small groups (little data) → shrink more toward the grand mean
- Large groups (lots of data) → the data speaks; shrink less
- Heterogeneous groups (large σ) → less shrinkage overall
- Homogeneous groups (small σ) → more shrinkage overall

### 4. Two levels of the model
- **Level 1** (within group): `S_i ~ Binomial(N_i, p_i)`,  `logit(p_i) = α[tank_i]`
- **Level 2** (between groups): `α[tank] ~ Normal(ᾱ, σ)` — the distribution of group parameters

You can add more levels, more predictors at each level, and varying slopes — that's where Chapter 13 goes next.

---
*Chapter 13 — Statistical Rethinking (McElreath, 2nd ed.)*